### Import Libraries

In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))


In [ ]:
import torch
from torch import nn

from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoProcessor

from src.model import WhisperAccentConfig, WhisperAccentForConditionalGeneration, register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import model_init, ModelArguments, WhisperAccentTrainingArguments
from src.train.trainer import WhisperAccentTrainer

register_whisper_accent()


### Load Model and Processor

In [ ]:
MODEL_TYPE = "whisper_accent"
BASE_MODEL_NAME_OR_PATH = "openai/whisper-small.en"
IS_MULTILINGUAL = False
DATASET_NAME="westbrook/English_Accent_DataSet"

In [ ]:
model_args = ModelArguments(
    model_type=MODEL_TYPE,
    base_model_name_or_path=BASE_MODEL_NAME_OR_PATH,
    is_multilingual=IS_MULTILINGUAL,
)


In [ ]:
# Load model and processor; whisper / whisper_accent models are supported
if model_args.model_type == "whisper_accent":
    processor = WhisperProcessor.from_pretrained(model_args.base_model_name_or_path)
    model = model_init(model_args)
elif model_args.model_type == "whisper":
    processor = WhisperProcessor.from_pretrained(model_args.base_model_name_or_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_args.base_model_name_or_path)
    # Update generation config; https://github.com/openai/whisper/discussions/2094
    if model.generation_config.is_multilingual:
        model.generation_config.language = "en"
        model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
else:
    raise ValueError(f"Invalid model type: {model_args.model_type}")


### Load Dataset and DataCollator

In [ ]:
dataset = WhisperDataset(
    data_path=DATASET_NAME,
    split="test",
    processor=processor,
    multilingual_model=False,
    num_proc=16,
)

collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    return_accent_labels=MODEL_TYPE == "whisper_accent",
)


In [ ]:
batch = collator([dataset[i] for i in range(4)])


### Inference

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(
        **batch,
        return_dict=True,
    )
    logits = outputs.logits
